# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale

## RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products

#### For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals - and we will give it a hand.

We discovered that LLMs are really good at this, out of the box.

And we discovered that we can beat a frontier LLM by fine-tuning an open-source LLM.

Now we are going to try **inference time** techniques instead of training -- by using RAG!

In [1]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [2]:
# environment

load_dotenv(override=True)
DB = "products_vectorstore"

In [3]:
# Log in to HuggingFace
# If you don't have a HuggingFace account, you can set one up for free at www.huggingface.co
# And then add the HF_TOKEN to your .env file as explained in the project README

hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

[HF] login bypassed


In [4]:
LITE_MODE = False

In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Now create a Chroma Datastore

Now we will use the free, open-source Vector database Chroma.  
We will create a Chroma datastore with 400,000 products from our training dataset.

In [6]:
client = chromadb.PersistentClient(path=DB)

# Introducing the SentenceTransformer Encoding LLM

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

In [7]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [8]:
# Pass in a list of texts, get back a numpy array of vectors

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)
Out[1]: 
array([-5.68234101e-02, -6.70465454e-02,  4.41129692e-02,  5.98603534e-03,
       -2.28949264e-02, -2.95300409e-02,  5.56369908e-02,  3.42665575e-02,
       -1.08529374e-01, -3.81691083e-02, -7.43872598e-02, -1.03664257e-01,
        1.69148073e-02,  1.33924722e-03, -6.86191395e-02,  8.99353921e-02,
       -1.45186391e-02, -2.43884884e-02,  4.21853038e-03, -9.62992385e-02,
       -2.51799338e-02,  4.60676327e-02,  4.95346822e-03, -3.88679579e-02,
        1.07717933e-03,  6.82337508e-02, -1.13859670e-02, -5.83416633e-02,
       -1.03801303e-02, -1.74953043e-02, -1.86478589e-02,  4.07058652e-03,
        1.59438066e-02,  6.49722442e-02,  3.71175818e-02,  2.78225169e-02,
       -4.41945195e-02, -2.34372765e-02,  9.71035361e-02, -5.06139062e-02,
       -1.93864517e-02, -3.83471437e-02,  4.76066805e-02, -3.36106345e-02,
        5.08286878e-02,  3.57935131e-02,  2.91812420e-03, -1.06529154e-01,
        4.07211818e-02, -5.85452130e-04, -1.05607502e-01, -1.03584349e-01,
        3

## With that background, let's populate our Chroma database

### By calculating vectors for 800,000 scraped products

This takes 30 minutes on my machine on my GPU - it might take longer for you - feel free to use the Lite dataset!

In [9]:
# Check if the collection exists; if not, create it

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

# Let's visualize the vectorized data

In [10]:
# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!

MAXIMUM_DATAPOINTS = 10_000

In [11]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [12]:
# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [13]:
# Let's try a 2D chart
# TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [14]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [15]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [16]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [17]:
test[0]

Out[1]: <Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal = $219.0>


In [18]:
def vector(item):
    return encoder.encode(item.summary)

In [19]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [20]:
find_similars(test[0])

Out[1]: 
(['HyperX QuadCast S USB Condenser Microphone with RGB lighting',
  'Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones',
  'Logitech MX Master 3S Wireless Performance Mouse',
  'Apple iPad Air 10.9-inch 64GB Wi-Fi (Latest Model)',
  'Nintendo Switch OLED Model with White Joy-Con'],
 [119.0, 228.0, 89.0, 499.0, 309.0])


In [21]:
# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions

def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [22]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
HyperX QuadCast S USB Condenser Microphone with RGB lighting
Price is $119.00

Potentially related product:
Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones
Price is $228.00

Potentially related product:
Logitech MX Master 3S Wireless Performance Mouse
Price is $89.00

Potentially related product:
Apple iPad Air 10.9-inch 64GB Wi-Fi (Latest Model)
Price is $499.00

Potentially related product:
Nintendo Switch OLED Model with White Joy-Con
Price is $309.00




In [23]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [24]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

Estimate the price of this product. Respond with the price, no explanation

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
HyperX QuadCast S USB Condenser Microphone with RGB lighting
Price is $119.00

Potentially related product:
Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones
Price is $228.00

Potentially related product:
Logitech MX Master 3S Wireless Performance Mouse
Price is $89.00

Potentially related product:
Apple iPad Air 10.9-inch 64GB Wi-F

In [25]:
# The function for gpt-5-mini

def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [26]:
# How much does our favorite distortion pedal cost?

test[0].price

Out[1]: 219.0


In [27]:
# Let's do this!!

gpt_5__1_rag(test[0])

Out[1]: 'Price is $299.00'


In [28]:
evaluate(gpt_5__1_rag, test)

$80 $183 $244 

  0%|          | 0/3 [00:00<?, ?it/s]

In [29]:
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [30]:
def specialist(item):
    return pricer.price.remote(item.summary)


In [31]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Download the Neural Network weights from Week 6 into this directory

The file `deep_neural_network.pth` here:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [32]:

from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

def deep_neural_network(item):
    return runner.inference(item.summary)

In [33]:
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


In [34]:
evaluate(ensemble, test)

$65 $168 $218 

  0%|          | 0/3 [00:00<?, ?it/s]

In [35]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [36]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

---------------------------------------------------------------------------
APIStatusError                            Traceback (most recent call last)
Cell In[1], line 4
      1 from agents.frontier_agent import FrontierAgent
      3 agent = FrontierAgent(collection)
----> 4 agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

File ~\projects\llm_engineering\week8\agents\frontier_agent.py:87, in FrontierAgent.price(self, description)
     83 documents, prices = self.find_similars(description)
     84 self.log(
     85     f"Frontier Agent is about to call {self.MODEL} with context including 5 similar products"
     86 )
---> 87 response = self.client.chat.completions.create(
     88     model=self.MODEL,
     89     messages=self.messages_for(description, documents, prices),
     90     seed=42,
     91     reasoning_effort="none",
     92 )
     93 reply = response.choices[0].message.content
     94 result = self.get_price(reply)


Batches: 100%|##########| 1/1 [00:00<00:00, 18.46it/s]


In [37]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

---------------------------------------------------------------------------
APIStatusError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

File ~\projects\llm_engineering\week8\agents\frontier_agent.py:87, in FrontierAgent.price(self, description)
     83 documents, prices = self.find_similars(description)
     84 self.log(
     85     f"Frontier Agent is about to call {self.MODEL} with context including 5 similar products"
     86 )
---> 87 response = self.client.chat.completions.create(
     88     model=self.MODEL,
     89     messages=self.messages_for(description, documents, prices),
     90     seed=42,
     91     reasoning_effort="none",
     92 )
     93 reply = response.choices[0].message.content
     94 result = self.get_price(reply)

File ~\projects\llm_engineering\.venv\Lib\site-packages\openai\_utils\_utils.py:286, in required_args.<locals>.inner

Batches: 100%|##########| 1/1 [00:00<00:00, 24.71it/s]


In [38]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()


In [39]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

Out[1]: 228.18450927734375


In [40]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [41]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

File ~\projects\llm_engineering\week8\agents\ensemble_agent.py:33, in EnsembleAgent.price(self, description)
     25 """
     26 Run this ensemble model
     27 Ask each of the models to price the product
   (...)     30 :return: an estimate of its price
     31 """
     32 self.log("Running Ensemble Agent - preprocessing text")
---> 33 rewrite = self.preprocessor.preprocess(description)
     34 self.log(f"Pre-processed text using {self.preprocessor.model_name}")
     35 specialist = self.specialist.price(rewrite)

File ~\projects\llm_engineering\week8\agents\preprocessor.py:45, in Preprocessor.preprocess(self, text)
     38 messages = self.messages_for(text)
     39 response = completion(
     40     messages=messages,
   